# Domain 5 · Context Management & Reliability (15%)

**How to use this notebook.** One section per task statement (5.1 → 5.6), in guide
order. Each section: the **verbatim guide quote** → a **plain-English table** → a
**runnable cell** that makes the concept observable → **anti-patterns** as commented
code → a **pointer to your own code** (`file:line`) → a **self-check**.

**Real calls, per-section best fit.** D5 splits in two: single-turn Claude *decisions*
(5.1 recall, 5.2 escalate-vs-resolve, 5.5 per-field confidence, 5.6 provenance synthesis)
use the **raw Messages API** (`client.messages.create`) — the same mechanism as your
`EX1`/`EX3`/`coordinator.py`. The genuinely *multi-agent* sections (5.3 error propagation,
5.4 subagent delegation) use the **Agent SDK** (`query()` + `Task`, `permission_mode=
"default"` + explicit `allowed_tools`, no bypass). Every runnable cell makes a **real**
call; only the *backend behind a tool* (a billing record, a canned search result) is a
fixture — never the mechanism under test.

> Setup: uses the `anthropic` SDK + the Agent SDK and your `ANTHROPIC_API_KEY` from the one
> `.env` in `claude-with-anthropic-api/`. Default model is **`claude-haiku-4-5`** — cheap, and
> these teach the *mechanism*, not model IQ. The model is read once from the `CLAUDE_MODEL` env
> var (Haiku is the fallback) and reused by **every** cell, raw-API and Agent SDK alike — so
> adding `CLAUDE_MODEL=claude-sonnet-4-6` to that one `.env` switches the whole notebook, no
> code change.

In [ ]:
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv
from pathlib import Path
import os, json, tempfile

# Load the key from your ONE .env (the VS Code kernel does NOT source your shell).
_candidates = [
    Path.cwd().parents[1] / "claude-with-anthropic-api" / ".env",
]
_found = find_dotenv(filename='.env', usecwd=True)   # nearest .env walking up (portable)
_envfile = Path(_found) if _found else next((p for p in _candidates if p.exists()), None)
if _envfile:
    load_dotenv(_envfile); print(f"Loaded .env from: {_envfile}")
else:
    print("WARNING: no .env found in", [str(p) for p in _candidates])
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set — check .env path above."

client = Anthropic()
# One switch for the whole notebook: set CLAUDE_MODEL in .env to override; else Haiku (cheap).
MODEL = os.environ.get("CLAUDE_MODEL") or "claude-haiku-4-5"
print("Using model:", MODEL)

# Anchor paths to the ccaf-prep dir so the 5.4 cell can explore your real tree.
REPO = next((p for p in Path.cwd().resolve().parents if (p / "claude-with-anthropic-api").exists()), Path.cwd())
CCAF_DIR = (REPO / "ccaf-prep") if (REPO / "ccaf-prep").exists() else Path.cwd()
print("ccaf-prep dir:", CCAF_DIR)

---
## 5.1 · Manage conversation context to preserve critical information across long interactions

> **Task Statement 5.1: Manage conversation context to preserve critical information across long interactions**
>
> **Knowledge of:**
> - Progressive summarization risks: condensing numerical values, percentages, dates, and customer-stated expectations into vague summaries
> - The "lost in the middle" effect: models reliably process information at the beginning and end of long inputs but may omit findings from middle sections
> - How tool results accumulate in context and consume tokens disproportionately to their relevance (e.g., 40+ fields per order lookup when only 5 are relevant)
> - The importance of passing complete conversation history in subsequent API requests to maintain conversational coherence
>
> **Skills in:**
> - Extracting transactional facts (amounts, dates, order numbers, statuses) into a persistent "case facts" block included in each prompt, outside summarized history
> - Extracting and persisting structured issue data (order IDs, amounts, statuses) into a separate context layer for multi-issue sessions
> - Trimming verbose tool outputs to only relevant fields before they accumulate in context (e.g., keeping only return-relevant fields from order lookups)
> - Placing key findings summaries at the beginning of aggregated inputs and organizing detailed results with explicit section headers to mitigate position effects
> - Requiring subagents to include metadata (dates, source locations, methodological context) in structured outputs to support accurate downstream synthesis
> - Modifying upstream agents to return structured data (key facts, citations, relevance scores) instead of verbose content and reasoning chains when downstream agents have limited context budgets

**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| Progressive summarization risk | Each time you compress history, exact **numbers/dates/expectations** blur into "a few hundred dollars". Once condensed, the precise value is gone — you can't recover `$337.50` from "a few hundred". |
| "Lost in the middle" | Models attend reliably to the **start** and **end** of a long input, less to the **middle**. A finding buried mid-context may be skipped. |
| Tool results bloat context | One order lookup returns 40+ fields; only ~5 matter. They pile up and crowd out what's relevant, token-for-token. |
| Pass complete history | Each API call is **stateless** — you resend the conversation every turn. Drop it and coherence breaks. |
| **"Case facts" block** | Pull transactional facts (amounts, order #s, dates, statuses) into a small block included **verbatim in every prompt, outside the summarized history** — so summarization can never blur them. |
| Trim before it accumulates | Keep only the relevant fields from a verbose tool output *before* it enters context (your `EX3` `trim_for_downstream`). |
| Key findings **first** + headers | Put the summary at the **top** and use explicit section headers to fight position effects. |

**Run it — one realistic flow, one tool (REAL calls).** Claude is given a single tool,
`lookup_order`, and **decides to call it** (a minimal agentic loop, D1.1). We **trim** its
40-field result to the ~4 relevant fields and **persist** them into a real, stable `case_facts`
dict — that dict *is* the case-facts block, kept verbatim *outside* the summarized history.
Then the payoff 5.1 is really about: we ask Claude for the exact figures **(a)** over the
progressively-summarized history alone (the number blurred to "a few hundred") vs **(b)** with
the case-facts block prepended. Only **(b)** preserves them. The only fixtures are the record
behind the tool and the toy summary; every model call is real.

*(Out of scope here, by design: the "ask again until you have it" gate is D1.4; structured
extraction of customer-stated facts is D4.3 / `EX3`.)*

In [ ]:
# 5.1 — ONE flow in TWO phases. Raw Messages API on purpose: this is a single-turn decision, and per
#        the notebook's convention multi-agent work uses the Agent SDK (5.3/5.4). The layer follows the
#        concept — raw API exposes the primitive (tool_use/tool_result); the SDK is for runtime features.
#   PHASE 1 (earlier in the conversation, ~turn 2): Claude calls a tool; we trim + persist the fact
#           into the case-facts block.
#   PHASE 2 (now, ~turn 14): the SAME question over two context shapes — the block survives, the
#           summary doesn't. We do NOT rebuild the block here; the point is you persisted it once.
# Fixtures = the 40-field record behind the tool + the toy summary; every Claude call below is REAL.

# THE TOOL Claude can call. The 40+ field record behind it is the only fixture.
LOOKUP_ORDER = {"name": "lookup_order",
    "description": "Fetch the full order record by order id (amount, date, status, plus metadata).",
    "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}},
                     "required": ["order_id"]}}
def run_lookup(order_id):                           # fixture backend (a real DB/API call in production)
    return {"id": order_id, "amount": 337.50, "date": "2024-03-03", "status": "delivered-damaged",
            **{f"meta_{i}": "irrelevant" for i in range(40)}}       # 40+ fields; only ~4 matter

# THE CASE-FACTS BLOCK: a real persistent dict, OUTSIDE the summarized history. The tool result is
# TRIMMED before it lands here (the 5.1 skill) — we never carry 40 fields into context.
case_facts = {}
KEEP = {"id": "order_id", "amount": "refund_amount", "date": "date", "status": "status"}

# PHASE 1 — raw-API tool use has TWO halves, and Claude only does the first:
#   (1) Claude DECIDES and returns a `tool_use` block (name + args). THAT is "Claude calling the tool".
#   (2) YOUR code executes the function and returns a `tool_result`. The API never runs your function.
# That tool_use -> execute -> tool_result loop IS the protocol (same DISPATCH idea as your agent.py).
msgs = [{"role": "user", "content": "Look up order A-1001 so we have its details on file."}]
while True:
    resp = client.messages.create(model=MODEL, max_tokens=400, tools=[LOOKUP_ORDER], messages=msgs)
    if resp.stop_reason != "tool_use":             # D1.1 loop: continue on tool_use, stop otherwise
        break
    msgs.append({"role": "assistant", "content": resp.content})
    out = []
    for b in resp.content:
        if b.type == "tool_use" and b.name == "lookup_order":     # (1) Claude's request to call it
            rec = run_lookup(b.input["order_id"])                  # (2) WE execute it; 40+ fields back...
            trimmed = {dst: rec[src] for src, dst in KEEP.items() if rec.get(src) is not None}
            case_facts.update(trimmed)                             # ...only the ~4 relevant ones persist
            out.append({"type": "tool_result", "tool_use_id": b.id,
                        "content": json.dumps(trimmed)})           # (2) return the result so Claude continues
    msgs.append({"role": "user", "content": out})

def case_facts_block():                             # the dict IS the block: verbatim, never summarized
    return ("CASE FACTS (verified, authoritative — trust these over any summary):\n"
            + json.dumps(case_facts, indent=2))
print("Claude called lookup_order; trimmed result persisted into the block:\n" + case_facts_block())

# PHASE 2 (what 5.1 tests): many turns later, the OLD turns got progressively summarized — and that
# BLURRED the number. We do NOT re-call the tool; we only ask whether the persisted block survives.
FILLER = "\n".join(f"  turn {i}: small talk about weather and shipping times." for i in range(1, 14))
summarized_old = ("CONVERSATION SUMMARY (older turns, progressively summarized):\n" + FILLER +
    "\n  ...customer agreed to a refund of a few hundred dollars on order A-1001...\n" + FILLER)

Q = "What is the EXACT refund amount, order number, and date? Answer in one line."
def ask(context):
    r = client.messages.create(model=MODEL, max_tokens=120,
        messages=[{"role": "user", "content": context + "\n\nQUESTION: " + Q}])
    return "".join(b.text for b in r.content if b.type == "text").strip()

print("\n(a) summary only          ->", ask(summarized_old))                              # number is gone
print("(b) case-facts + summary  ->", ask(case_facts_block() + "\n\n" + summarized_old))  # survives
print("\n-> Same question, two context shapes. The block was BUILT by trimming a real tool result and\n"
      "   kept verbatim outside the summary, so $337.50 / A-1001 / 2024-03-03 survive (b) but not (a).")

**The anti-patterns (exam distractors).** From the mapping: *progressive summarization
of numbers.* As code:

In [ ]:
# ANTI-PATTERN 1: let progressive summarization condense the NUMBERS.
#   history = summarize(history)   # '$337.50' -> 'a few hundred' -> the exact value is unrecoverable
#   -> Keep amounts/dates/order#s in a verbatim CASE-FACTS block, outside the summary.

# ANTI-PATTERN 2: bury the key finding in the MIDDLE of a long aggregated input.
#   prompt = filler + key_finding + filler   # lost-in-the-middle -> the model may skip it
#   -> Put key findings FIRST, with explicit section headers.

# ANTI-PATTERN 3: dump the whole 40-field tool result into context every turn.
#   messages.append(full_order_json)   # tokens accumulate disproportionately to relevance
#   -> Trim to the ~5 relevant fields BEFORE it enters context.

# ANTI-PATTERN 4: drop conversation history to 'save tokens'.
#   messages = [latest_user_turn]   # stateless API -> coherence collapses
#   -> Resend complete history (summarized OLD turns + verbatim case-facts).
print("Correct: verbatim CASE-FACTS block (numbers/dates) outside the summary; key findings first; "
      "trim verbose tool outputs to relevant fields; resend complete history.")

**In your own code — you already wrote this.** Two anchors:

- **Exercise 3:** `ccaf-prep/exercises/03-extraction-pipeline/extract.py` — `trim_for_downstream`
  (**lines 234–241**) keeps only the relevant fields and **drops the verbose `confidence`
  blob** before passing forward (the comment cites D5.1 explicitly). That's "trim verbose
  tool outputs" as real code.
- **Exercise 1:** `ccaf-prep/exercises/01-support-agent/tools.py` — `escalate_to_human`
  (**lines 78–81**) returns a structured **`handoff_summary`** because "the human has NO
  access to the transcript" — i.e. the transactional facts are persisted *outside* the
  conversation, the same instinct as a case-facts block.

Open `extract.py:234` and map each kept field to "only relevant fields" in the table.

**Self-check** (cover the answers)

1. A customer agreed to a `$337.50` refund 20 turns ago. How do you guarantee that exact number survives summarization?
2. You aggregate 8 subagent findings into one prompt; one is critical. Where do you put it, and why?
3. An order lookup returns 40+ fields; 5 matter. What do you do before it enters context?
4. Why must you resend the whole conversation history each API call?

<details><summary>answers</summary>

1. Keep it in a verbatim **CASE-FACTS block** included in every prompt, *outside* the summarized history — summarization can't blur what it never touches.
2. **First** (top), with an explicit section header — to beat the **lost-in-the-middle** position effect.
3. **Trim** to the ~5 relevant fields (`trim_for_downstream`); verbose tool outputs consume tokens disproportionately to relevance.
4. The Messages API is **stateless** — it has no memory of prior turns; omitting history breaks conversational coherence.
</details>

---
## 5.2 · Design effective escalation and ambiguity resolution patterns

> **Task Statement 5.2: Design effective escalation and ambiguity resolution patterns**
>
> **Knowledge of:**
> - Appropriate escalation triggers: customer requests for a human, policy exceptions/gaps (not just complex cases), and inability to make meaningful progress
> - The distinction between escalating immediately when a customer explicitly demands it versus offering to resolve when the issue is straightforward
> - Why sentiment-based escalation and self-reported confidence scores are unreliable proxies for actual case complexity
> - How multiple customer matches require clarification (requesting additional identifiers) rather than heuristic selection
>
> **Skills in:**
> - Adding explicit escalation criteria with few-shot examples to the system prompt demonstrating when to escalate versus resolve autonomously
> - Honoring explicit customer requests for human agents immediately without first attempting investigation
> - Acknowledging frustration while offering resolution when the issue is within the agent's capability, escalating only if the customer reiterates their preference
> - Escalating when policy is ambiguous or silent on the customer's specific request (e.g., competitor price matching when policy only addresses own-site adjustments)
> - Instructing the agent to ask for additional identifiers when tool results return multiple matches, rather than selecting based on heuristics

**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| Escalation **triggers** | Escalate on: (1) explicit request for a human, (2) **policy gap/exception** — not merely "hard", (3) no meaningful progress. |
| Explicit demand → escalate **now** | "Get me a human" means hand off immediately — don't investigate first. |
| Straightforward → **offer to resolve** | If it's within your ability, acknowledge and solve; escalate only if they reiterate. |
| Sentiment ≠ complexity | Anger doesn't mean the case is hard. **Sentiment-based** escalation and **self-reported confidence** are bad proxies (the model is already wrongly confident on hard cases). |
| Multiple matches → **ask** | If a lookup returns 2 "John Smith"s, request another identifier — never heuristically pick one. |
| Explicit criteria + few-shot | The fix is **prompt-level**: spell out when to escalate vs resolve, with examples (sample Q3) — not a sentiment classifier or an ML model. |

> **Glossary — *few-shot*.** "Few-shot" = putting **a few worked examples** (`input -> decision`) in the prompt; "few" is the *count* (vs. *zero-shot* = none), not the polarity. It includes both **positive examples** ("asks for a human" -> escalate) and **counter-examples** ("furious but solvable" -> *don't* escalate) — the counter-example of the tempting case usually teaches more than another easy one. *Spell out* = state the rules **to the model, in the system prompt** (not to a person, not in code).

> **Why self-reported confidence is a bad proxy.** It's not bad luck on one isolated edge case: the LLM's miscalibration is **correlated with difficulty** — the model is confident *exactly where it's wrong*. The proxy fails **precisely on the hard cases** it's meant to catch, so tests on easy cases can look well-calibrated and still fail on the tail that matters.

**Run it — escalate vs resolve, three real decisions.** Same policy (explicit criteria +
few-shot, mirroring `EX1`), three messages, one real call each. We give Claude an
`escalate_to_human` tool and read **whether it calls it** — the decision is Claude's, not
a keyword switch. Watch: explicit human request → escalate; frustrated-but-solvable →
resolve (sentiment ≠ complexity); policy-gap price-match → escalate.

In [ ]:
# 5.2 — REAL decisions: we only read WHETHER Claude calls escalate_to_human. Decision is Claude's.
ESCALATE_TOOL = {"name": "escalate_to_human",
    "description": "Hand the case to a human agent. Use for explicit human requests, policy "
                   "gaps/exceptions, or when you cannot make progress.",
    "input_schema": {"type": "object", "properties": {"reason": {"type": "string"}},
                     "required": ["reason"]}}

POLICY = (
    "You are a support agent. RESOLVE straightforward issues yourself. ESCALATE (call "
    "escalate_to_human) ONLY when: (1) the customer explicitly asks for a human, (2) our policy "
    "is silent or ambiguous on the request, or (3) you cannot make progress. Do NOT escalate "
    "merely because the customer is frustrated if the issue is within your ability. When a "
    "request falls OUTSIDE the Known policies, do not decide it yourself -- neither grant nor "
    "refuse it -- escalate so a human can rule on the exception.\n"
    "Known policies (this is the FULL list -- anything not covered here is a policy gap):\n"
    "- Order tracking and status updates.\n"
    "- Free replacement for items that arrive damaged (with a photo).\n"
    "- Price adjustments ONLY for price drops on OUR OWN site within 14 days of purchase.\n"
    "Examples:\n"
    "- 'Get me a human now' -> escalate immediately (explicit request).\n"
    "- 'My order is late and I am furious' -> resolve (offer tracking/replacement); frustration "
    "alone is not a reason to escalate.\n"
    "- 'Do you match a COMPETITOR's price?' -> escalate: our price policy covers only our own "
    "site, so competitor matching is a policy gap; do not answer yes or no yourself -- a human "
    "must decide whether to make an exception.")

def decide(msg):
    r = client.messages.create(model=MODEL, max_tokens=200, system=POLICY,
        tools=[ESCALATE_TOOL], messages=[{"role": "user", "content": msg}])
    escalated = any(b.type == "tool_use" and b.name == "escalate_to_human" for b in r.content)
    text = "".join(b.text for b in r.content if b.type == "text").strip()
    return escalated, text

CASES = [
    ("explicit human request", "I don't want a bot. Connect me to a human agent right now."),
    ("frustrated but solvable", "This is ridiculous, my order is 2 days late and I'm furious!"),
    ("policy gap (price match)", "Competitor X sells this $20 cheaper. Will you match their price?")]
for label, msg in CASES:
    esc, text = decide(msg)
    tail = "" if esc else "  resolves: " + text[:60]
    print(f"[{label:24}] -> {'ESCALATE' if esc else 'RESOLVE'}{tail}")
print("\n-> Escalate on EXPLICIT request and POLICY GAP; RESOLVE on mere frustration "
      "(sentiment != complexity). The criteria drive Claude's decision, not a keyword rule.")

**The anti-patterns (exam distractors).** From the mapping: *#5 sentiment ≠ complexity.*
As code:

In [ ]:
# ANTI-PATTERN 1 (sample Q3 D): escalate on SENTIMENT.
#   if sentiment(msg) < -0.5: escalate()   # anger != hard case; escalates easy-but-angry, keeps hard-but-calm
#   -> Escalate on explicit request / policy gap / no progress, not mood.

# ANTI-PATTERN 2 (sample Q3 B): trust the model's SELF-REPORTED confidence.
#   if model_confidence < 7: escalate()    # LLMs are miscalibrated — already wrongly confident on hard cases
#   -> Self-confidence is an unreliable proxy for complexity.

# ANTI-PATTERN 3 (sample Q3 C): train a separate ML CLASSIFIER first.
#   route = trained_classifier.predict(ticket)   # labeled data + infra before trying the prompt fix
#   -> Over-engineered; add explicit criteria + few-shot to the prompt first.

# ANTI-PATTERN 4: heuristically PICK one of multiple customer matches.
#   customer = matches[0]                  # silently guesses identity -> wrong account, wrong refund
#   -> Ask for an additional identifier when a lookup returns multiple matches.
print("Correct (sample Q3 = A): explicit escalation criteria + few-shot in the system prompt; "
      "honor explicit requests immediately; ask for an ID on multi-match.")

**In your own code — you already wrote this.** Two anchors:

- **Exercise 1:** `ccaf-prep/exercises/01-support-agent/agent.py` — the `SYSTEM` prompt
  (**lines 32–38**) states the escalation rule ("Refunds over $500 must be escalated to a
  human… include a full handoff summary"); the gate (**lines 52–54**) *forces* escalation
  on the over-limit case, and `tools.py` `escalate_to_human` (**78–81**) is the handoff.
- **Exercise 1 (ambiguity):** `ccaf-prep/exercises/01-support-agent/tools.py` — `get_customer`
  (**lines 38–55**) returns **two "John Smith" records** (**lines 22–23**) and, on multiple
  matches (**line 50+**), asks for more identifiers instead of guessing — D5.2 ambiguity
  resolution, exercised by `agent.py:113`.

Open `agent.py:32` and map each clause to an escalation **trigger** in the table.

**Self-check** (cover the answers)

1. A calm customer has a genuinely hard policy-exception case; an angry one has a simple late order. Which escalates, and why?
2. The cheapest first fix for poor escalation calibration — sentiment analysis, a trained classifier, or what? (sample Q3)
3. `get_customer("John Smith")` returns two records. What do you do?
4. "I demand a human." Investigate first, or escalate immediately?

<details><summary>answers</summary>

1. The **calm policy-exception** case escalates (policy gap = a real trigger); the angry late-order is **resolved** (sentiment ≠ complexity).
2. **Explicit escalation criteria + few-shot examples** in the system prompt (Q3 = A). Sentiment (D) and self-confidence (B) are bad proxies; a classifier (C) is over-engineered.
3. **Ask for an additional identifier** — never heuristically pick one (wrong account → wrong refund).
4. **Escalate immediately** — honor an explicit request without first attempting investigation.
</details>

---
## 5.3 · Implement error propagation strategies across multi-agent systems

> **Task Statement 5.3: Implement error propagation strategies across multi-agent systems**
>
> **Knowledge of:**
> - Structured error context (failure type, attempted query, partial results, alternative approaches) as enabling intelligent coordinator recovery decisions
> - The distinction between access failures (timeouts needing retry decisions) and valid empty results (successful queries with no matches)
> - Why generic error statuses ("search unavailable") hide valuable context from the coordinator
> - Why silently suppressing errors (returning empty results as success) or terminating entire workflows on single failures are both anti-patterns
>
> **Skills in:**
> - Returning structured error context including failure type, what was attempted, partial results, and potential alternatives to enable coordinator recovery
> - Distinguishing access failures from valid empty results in error reporting so the coordinator can make appropriate decisions
> - Having subagents implement local recovery for transient failures and only propagate errors they cannot resolve, including what was attempted and partial results
> - Structuring synthesis output with coverage annotations indicating which findings are well-supported versus which topic areas have gaps due to unavailable sources

**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| **Structured error context** | An error the coordinator can *reason* about: `failure_type`, `attempted_query`, `partial_results`, `suggested_alternative` — not just "failed". |
| access failure vs **empty result** | "Timed out" (an **access_failure** → maybe retry/fallback) is NOT "ran fine, 0 matches" (a **valid empty result** → a real answer). Don't conflate them. |
| Generic status hides context | `"search unavailable"` strips away everything the coordinator needs to recover intelligently. |
| Suppressing / killing are both wrong | Returning an error as empty-success **hides** the failure; terminating the whole workflow on one subagent failure is **over-reacting**. |
| Local recovery, propagate the rest | A subagent retries its own transient blips; it escalates only what it **can't** fix — with what it attempted + partial results. |
| Coverage annotations | Synthesis says which areas are **well-supported** vs which have **gaps** because a source was unavailable. |

**Run it — structured vs generic error, real coordinator recovery (Agent SDK).** A
coordinator (`query()`) calls a subagent's `population_db` tool, which **fails** (simulated
outage). We toggle what the failing tool returns: a **structured** error (`failure_type` +
`partial_results` + `suggested_alternative`) vs a **generic** "database unavailable". We
force the coordinator to rely on the **tool only** (no outside knowledge), so the contrast
is clean: it salvages the structured case from the cached partial and reports a gap on the
generic one. The fixture is the failing backend; the **recovery decision is Claude's**.
(D5.3 / sample Q8.)

In [ ]:
# 5.3 — REAL Agent SDK: a coordinator calls a data tool, the tool FAILS, and it must RECOVER
# from whatever the error carries. Fixture = the failing backend; the recovery decision is Claude's.
from claude_agent_sdk import (query, tool, create_sdk_mcp_server, ClaudeAgentOptions,
                              AssistantMessage, ToolUseBlock, ResultMessage)

STRUCTURED = True   # toggled by run_coord()

@tool("population_db", "Look up a city's population from the research database.", {"city": str})
async def population_db(args):
    # The lookup FAILS (simulated outage). Return a STRUCTURED vs GENERIC error per the toggle.
    if STRUCTURED:
        # D5.3 — structured error context the coordinator can actually act on.
        return {"content": [{"type": "text", "text": json.dumps({
            "isError": True, "failure_type": "access_failure", "attempted_query": args["city"],
            "partial_results": {"cached_value": "37.0M", "as_of": "2023 (stale cache)"},
            "suggested_alternative": "use the cached_value above, noting it is stale"})}],
            "is_error": True}
    return {"content": [{"type": "text", "text": "Error: database unavailable."}], "is_error": True}

research = create_sdk_mcp_server("research", tools=[population_db])
DB = "mcp__research__population_db"

async def run_coord(structured):
    global STRUCTURED
    STRUCTURED = structured
    opts = ClaudeAgentOptions(model=MODEL, mcp_servers={"research": research},
        allowed_tools=[DB], permission_mode="default", max_turns=6)   # real allow-list, no bypass
    prompt = ("You are a research coordinator. The ONLY source you may use is the population_db "
              "tool — do NOT use prior knowledge. Look up Tokyo's population. If the tool returns "
              "an error, RECOVER using ONLY what the error provides (partial/cached results or its "
              "suggested alternative); if it gives you nothing usable, reply that the data is "
              "currently unavailable. Answer in ONE sentence.")
    called, final = [], None
    async for m in query(prompt=prompt, options=opts):
        if isinstance(m, AssistantMessage):
            called += [b.name for b in m.content if isinstance(b, ToolUseBlock)]
        elif isinstance(m, ResultMessage):
            final = m.result
    return called, final

# Top-level await: Jupyter runs these directly.
c1, f1 = await run_coord(True)
print(f"STRUCTURED error (tools called: {c1}) ->", (f1 or "")[:200])
c2, f2 = await run_coord(False)
print(f"\nGENERIC error    (tools called: {c2}) ->", (f2 or "")[:200])
print("\n-> Structured context lets the coordinator recover (it uses the cached 37.0M partial); a "
      "generic 'unavailable' leaves it nothing, so it reports the gap. (D5.3 / sample Q8 = A)")

**The anti-patterns (exam distractors).** From the mapping: *suppress error / kill
workflow.* As code:

In [ ]:
# ANTI-PATTERN 1 (sample Q8 C): SUPPRESS the error — return empty as success.
#   except Timeout: return {"results": [], "ok": True}   # hides the failure; risks incomplete output
#   -> A timeout is an access_failure, NOT a valid empty result. Report it.

# ANTI-PATTERN 2 (sample Q8 D): KILL the whole workflow on one subagent failure.
#   except Timeout: raise SystemExit   # one flaky source nukes the entire research run
#   -> Propagate structured context UP; let the coordinator recover or proceed with partials.

# ANTI-PATTERN 3 (sample Q8 B): collapse to a GENERIC status.
#   return {"status": "search unavailable"}   # strips failure_type / attempted / partial / alternative
#   -> Return structured context so the coordinator can make an informed decision.

# ANTI-PATTERN 4: treat 'access_failure' and 'empty_result' identically.
#   if error: retry_forever(...)   # retrying a valid 'no matches' burns turns for nothing
#   -> Distinguish them; retry access failures, accept empty results.
print("Correct (sample Q8 = A): structured error context (failure_type + attempted + partial + "
      "alternative) so the coordinator can recover; never suppress, never kill the workflow.")

**In your own code — you already built this.** Anchor:

- **Exercise 4:** `ccaf-prep/exercises/04-multi-agent-research/coordinator.py` — `_err(...)`
  (**lines 64–68**) is the structured-error shape; `web_search` (**lines 72–81**) returns
  an **`access_failure`** (with a cached **partial** + a **`suggested_alternative`**) for an
  "outage" query but an **`empty_result`** for "unicorn" — the exact access-vs-empty
  distinction. `research_subagent` (**131–132**) **propagates** the structured error up
  *without suppressing*, and the coordinator (**191–200**) surfaces it and **keeps going**.
  The `__main__` block (**256–263**) prints both failure types.

Run `uv run python coordinator.py` and read the `STRUCTURED ERROR propagation` section.

**Self-check** (cover the answers)

1. A web-search subagent times out. Which beats a generic "search unavailable" — and what four things go in it? (sample Q8)
2. "Timed out" vs "ran fine, 0 matches" — which may warrant a retry?
3. Two anti-patterns the guide names explicitly for a single subagent failure.
4. What is a "coverage annotation" in synthesis output?

<details><summary>answers</summary>

1. **Structured error context** (Q8 = A): `failure_type`, `attempted_query`, `partial_results`, `suggested_alternative`.
2. The **access failure** (timeout) may warrant a retry/fallback; the **valid empty result** is a real answer — don't retry it.
3. **Suppressing** the error (empty-as-success) and **terminating** the whole workflow on one failure.
4. A note on which findings are **well-supported** vs which topic areas have **gaps** because a source was unavailable.
</details>

---
## 5.4 · Manage context effectively in large codebase exploration

> **Task Statement 5.4: Manage context effectively in large codebase exploration**
>
> **Knowledge of:**
> - Context degradation in extended sessions: models start giving inconsistent answers and referencing "typical patterns" rather than specific classes discovered earlier
> - The role of scratchpad files for persisting key findings across context boundaries
> - Subagent delegation for isolating verbose exploration output while the main agent coordinates high-level understanding
> - Structured state persistence for crash recovery: each agent exports state to a known location, and the coordinator loads a manifest on resume
>
> **Skills in:**
> - Spawning subagents to investigate specific questions (e.g., "find all test files," "trace refund flow dependencies") while the main agent preserves high-level coordination
> - Having agents maintain scratchpad files recording key findings, referencing them for subsequent questions to counteract context degradation
> - Summarizing key findings from one exploration phase before spawning sub-agents for the next phase, injecting summaries into initial context
> - Designing crash recovery using structured agent state exports (manifests) that the coordinator loads on resume and injects into agent prompts
> - Using /compact to reduce context usage during extended exploration sessions when context fills with verbose discovery output

**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| Context degradation | In a long session the model drifts — it starts citing "typical patterns" instead of the specific class it actually found earlier. |
| **Scratchpad files** | Write key findings to a file; reference it later so the fact survives even when it scrolls out of context. |
| **Subagent delegation** | Send the *verbose* exploration ("find all test files") to a subagent with **isolated context**; the main agent keeps only the high-level summary. |
| **State manifest** for crash recovery | Each phase exports state to a known location; on resume the coordinator **loads the manifest** and injects it — no re-exploring from scratch. |
| Summarize between phases | Compress phase-1 findings before spawning phase-2 subagents; inject the summary as their initial context. |
| `/compact` | The Claude Code command that compresses a bloated session mid-exploration. |

**Run it — delegate focused explorations + checkpoint a manifest incrementally (Agent SDK).**
The main agent **delegates** each focused question to an `explorer` subagent via the **Task**
tool (scoped to `Grep`/`Glob`/`Read`); the subagent's verbose searching stays in its **isolated
context** while the coordinator keeps only the summary (D1.2). The **state manifest** is written
to a known location **after every phase** (and a phase is marked `in_progress` *before* it runs),
then once more at the end — so a crash mid-exploration leaves the last good checkpoint on disk and
**resume skips completed phases** (D5.4). Real `query()` over your real `ccaf-prep` tree; the
manifest is a real file.

In [ ]:
# 5.4 — REAL Agent SDK: delegate verbose exploration to subagents, then persist a manifest
# INCREMENTALLY so a crash mid-exploration resumes from the last good checkpoint (not from scratch).
from claude_agent_sdk import (query, ClaudeAgentOptions, AgentDefinition,
                              SystemMessage, ResultMessage)

EXPLORER = {"explorer": AgentDefinition(
    description="Investigates ONE focused question about the codebase.",
    prompt="Use Grep/Glob/Read to answer the ONE question, then report findings in 2-3 lines.",
    tools=["Grep", "Glob", "Read"], model=MODEL)}        # subagent scoped to read-only tools

async def delegate(question):
    opts = ClaudeAgentOptions(model=MODEL, agents=EXPLORER,
        allowed_tools=["Task", "Grep", "Glob", "Read"], permission_mode="default",
        cwd=str(CCAF_DIR), max_turns=8)                 # real allow-list, no bypass
    prompt = (f"You coordinate a large codebase exploration. DELEGATE this focused question to the "
              f"`explorer` subagent via the Task tool (keep your own context for high-level "
              f"coordination, not the raw search output): {question!r}. Then summarize its answer "
              f"in ONE sentence.")
    spawned, final = [], None
    async for m in query(prompt=prompt, options=opts):
        if isinstance(m, SystemMessage) and getattr(m, "subtype", None) == "task_started":
            spawned.append(m.data.get("subagent_type"))
            print(f"  [Task spawns subagent]: {m.data.get('subagent_type')!r} (ISOLATED context)")
        elif isinstance(m, ResultMessage):
            final = m.result
    return spawned, final

# D5.4 crash-recovery MANIFEST, written INCREMENTALLY. A real exploration has several phases; we
# checkpoint to a known location AFTER EACH ONE (and mark a phase 'in_progress' BEFORE it runs), so a
# crash mid-exploration leaves the last good state on disk — resume reloads it and skips done phases.
mdir = Path(tempfile.mkdtemp()); manifest = mdir / "manifest.json"
PHASES = {  # two focused phases over your real tree (enough to show incremental checkpointing)
    "confidence_schema": "Which exercise files define a per-field 'confidence' object in a tool input_schema?",
    "manifest_writers":  "Which Python files write or read a file named 'manifest.json'?"}

state = {"status": "in_progress",
         "phases": {k: {"status": "pending", "question": q, "summary": None, "delegated_to": None}
                    for k, q in PHASES.items()}}
def checkpoint():                                    # the dict IS the manifest: re-dumped on every update
    manifest.write_text(json.dumps(state, indent=2))

checkpoint()                                         # initial checkpoint: every phase 'pending'
for key, q in PHASES.items():
    if state["phases"][key]["status"] == "complete":          # RESUME: a prior run already did this phase
        print(f"  [{key}] already complete in manifest -> skip (no re-exploration)"); continue
    state["phases"][key]["status"] = "in_progress"; checkpoint()   # a crash HERE leaves 'in_progress' on disk
    spawned, summary = await delegate(q)
    state["phases"][key].update(status="complete", summary=(summary or "")[:160], delegated_to=spawned)
    checkpoint()                                     # checkpoint AFTER the phase completes
    print(f"  [{key}] complete -> {state['phases'][key]['summary']}")

state["status"] = "complete"; checkpoint()           # final manifest too (both incremental AND terminal)
print(f"\n  wrote manifest -> {manifest}")
resumed = json.loads(manifest.read_text())           # RESUME just reloads the file — no re-exploring
done = [k for k, v in resumed["phases"].items() if v["status"] == "complete"]
print(f"  RESUME loads manifest: status={resumed['status']!r}, completed phases={done}")
print("\n-> Delegation isolates verbose exploration in subagents; the manifest is checkpointed AFTER each "
      "phase (and once at the end),\n   so a crash resumes from the last checkpoint, not from scratch. (D5.4)")


**The anti-patterns (exam distractors).** From the mapping: *re-explore from scratch.*
As code:

In [ ]:
# ANTI-PATTERN 1: do all exploration in ONE long-lived context.
#   for q in many_questions: ctx += verbose_search(q)   # context degrades; model cites 'typical patterns'
#   -> Delegate each verbose probe to a subagent; keep only summaries in the main context.

# ANTI-PATTERN 2: keep findings only in CONTEXT (no scratchpad/manifest).
#   # 200 turns later the class you found is gone; you re-discover it (or hallucinate it)
#   -> Persist key findings to a scratchpad/manifest file and reference them.

# ANTI-PATTERN 3: write the manifest only ONCE, at the very end.
#   run_all_phases(); write_manifest(...)   # a crash mid-exploration leaves NOTHING (or stale state)
#   -> Checkpoint AFTER each phase (mark it 'in_progress' before) so a crash keeps the last good state.

# ANTI-PATTERN 4: on crash/restart, RE-EXPLORE the whole codebase from scratch.
#   restart(); explore_everything_again()   # throws away all prior progress
#   -> Load the state manifest on resume and inject it into prompts (skip completed phases).

# ANTI-PATTERN 5: let context fill with verbose discovery output and never /compact.
print("Correct: delegate verbose exploration to isolated subagents; persist findings to a "
      "scratchpad/manifest;\ncheckpoint after each phase; resume by LOADING the manifest; /compact a bloated session.")


**In your own code — you already built this.** Anchor:

- **Exercise 4:** `ccaf-prep/exercises/04-multi-agent-research/coordinator.py` — `write_manifest`
  (**lines 161–178**) exports `query`, selected subagents, an overall `status`, a per-phase
  `phase_status` (`pending`/`in_progress`/`complete`/`error`), `completed`, `errors`, and
  `findings` to **`manifest.json`** (a known location). The coordinator loop (**lines 192–211**)
  calls it **after every subagent** — marking a phase `in_progress` *before* it runs and writing a
  final `status="complete"` at the end — so a crash mid-run leaves the last good checkpoint and
  resume can skip phases already `complete`. The hub-and-spoke `subagent` (**lines 119–127**) gives
  each spoke an **isolated** `messages` list — verbose work stays out of the coordinator's context.
  The committed `manifest.json` is the real artifact.
- **Real Task delegation** is in `coordinator_sdk.py` (the SDK companion) — the cell above
  is the same mechanism over your own tree.

Open `manifest.json` and match each key to a "resume" field the table describes.

**Self-check** (cover the answers)

1. In a long exploration the model starts citing "typical patterns" instead of a class it found earlier. What's this called, and one fix?
2. Why delegate "find all test files" to a subagent instead of doing it inline?
3. The process crashes after phase 2. How do you resume without re-exploring everything?
4. Which Claude Code command compresses a bloated exploration session?

<details><summary>answers</summary>

1. **Context degradation** — fix with **scratchpad files** (persist key findings and reference them) and **subagent delegation**.
2. To isolate the **verbose** output in the subagent's context, keeping the main agent's context for high-level coordination.
3. Load the **state manifest** the coordinator exported and inject it into prompts — no re-exploration.
4. **`/compact`**.
</details>

---
## 5.5 · Design human review workflows and confidence calibration

> **Task Statement 5.5: Design human review workflows and confidence calibration**
>
> **Knowledge of:**
> - The risk that aggregate accuracy metrics (e.g., 97% overall) may mask poor performance on specific document types or fields
> - Stratified random sampling for measuring error rates in high-confidence extractions and detecting novel error patterns
> - Field-level confidence scores calibrated using labeled validation sets for routing review attention
> - The importance of validating accuracy by document type and field segment before automating high-confidence extractions
>
> **Skills in:**
> - Implementing stratified random sampling of high-confidence extractions for ongoing error rate measurement and novel pattern detection
> - Analyzing accuracy by document type and field to verify consistent performance across all segments before reducing human review
> - Having models output field-level confidence scores, then calibrating review thresholds using labeled validation sets
> - Routing extractions with low model confidence or ambiguous/contradictory source documents to human review, prioritizing limited reviewer capacity

**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| Aggregate masks segments | "97% overall" can hide a 70%-accurate **document type** or **field**. The average lies. |
| **Field-level** confidence | Have the model emit a confidence **per field**, not one number for the whole extraction — so you route attention precisely. |
| Calibrated with labeled sets | A raw model score isn't trustworthy until you tune the **threshold** against a labeled validation set. |
| **Stratified** sampling | Sample within each segment (type/field), not uniformly, to catch novel errors hiding in a small slice. |
| Validate by segment **before** automating | Prove accuracy holds per type/field before you reduce human review. |
| Route low-confidence / ambiguous | Send low-confidence or contradictory-source extractions to humans first — prioritize scarce reviewer time. |

**Run it — calibrate the threshold, then route per-field, then stratify (REAL calls).** One
block, end to end for 5.5: **(1)** *calibrate* a per-field confidence threshold **offline on a
labeled validation set** — tie it to *measured* accuracy, not vibes; **(2)** use that calibrated
threshold to **route** a clean vs a mixed source per field (the aggregate mean stays deceptively
high while the one weak field is the one sent to review); **(3)** remember an **aggregate accuracy
can hide a weak segment** — stratify by document type/field before automating. Every extraction is
a real `record_extraction` tool call (per-field confidence, like `EX3`).

In [ ]:
# 5.5 — Human review workflow + confidence CALIBRATION, end to end (REAL calls). ONE flow:
#   (1) CALIBRATE a per-field confidence threshold OFFLINE on a LABELED validation set, then
#   (2) use THAT calibrated threshold to ROUTE live extractions per field, then
#   (3) remember an AGGREGATE accuracy can hide a weak SEGMENT -> stratify by type/field.

# The extraction tool: per-field values + an HONEST per-field 0.0-1.0 confidence (like EX3).
REC_TOOL = {"name": "record_extraction",
    "description": "Record extracted refund fields WITH a per-field 0.0-1.0 confidence "
                   "(be honest: low when the source is vague or contradictory).",
    "input_schema": {"type": "object", "properties": {
        "amount": {"type": ["number", "null"]}, "date": {"type": ["string", "null"]},
        "order_id": {"type": ["string", "null"]},
        "confidence": {"type": "object", "properties": {
            "amount": {"type": "number"}, "date": {"type": "number"}, "order_id": {"type": "number"}},
            "required": ["amount", "date", "order_id"]}},
        "required": ["confidence"]}}

def extract(msg):
    r = client.messages.create(model=MODEL, max_tokens=300, tools=[REC_TOOL],
        tool_choice={"type": "tool", "name": "record_extraction"},
        messages=[{"role": "user", "content": "Extract the refund fields from:\n" + msg}])
    return next(b.input for b in r.content if b.type == "tool_use")

# ---- (1) CALIBRATION ------------------------------------------------------------------------
# A raw model score is just a GUESS until checked against KNOWN-correct answers. OFFLINE, before
# automating, run the extractor on a LABELED validation set and tie the threshold to MEASURED
# accuracy (re-run as data drifts). None = truth is 'unknown' (the model must ABSTAIN).
VALIDATION = [
    ("Refund $337.50 on order A-1001, dated 2024-03-03.",   {"amount": 337.50, "order_id": "A-1001", "date": "2024-03-03"}),
    ("Please refund order B-2002, $89.99, on 2024-05-12.",  {"amount": 89.99,  "order_id": "B-2002", "date": "2024-05-12"}),
    ("$1,250 back on order C-3003 from 2024-01-20.",        {"amount": 1250.0, "order_id": "C-3003", "date": "2024-01-20"}),
    ("Refund the order from last Tuesday, around three hundred bucks I think.",   # nothing precise ->
        {"amount": None, "order_id": None, "date": None}),                        # truth = abstain on all
    ("Order D-4004 -- refund half of the $200 I paid.",     {"amount": 100.0, "order_id": "D-4004", "date": None}),
    ("Refund $50 for order E5005 placed sometime in March.", {"amount": 50.0,  "order_id": "E5005", "date": None}),
]

def is_correct(field, pred, truth):
    blank = pred in (None, "", "UNKNOWN", "<UNKNOWN>", "N/A")     # treat sentinels as an abstention
    if truth is None: return blank                               # truth unknown -> right ONLY if abstained
    if blank: return False
    if field == "amount": return abs(float(pred) - truth) < 0.01
    return str(pred).strip() == str(truth)

samples = []  # (reported_confidence, was_it_ACTUALLY_right) for every field of every labeled row
for msg, truth in VALIDATION:
    ex = extract(msg)
    for f in ("amount", "date", "order_id"):
        samples.append((ex["confidence"][f], is_correct(f, ex.get(f), truth[f])))

# Sweep candidate thresholds; pick the LOWEST whose MEASURED accuracy meets the target
# (automate as much as stays safe; everything below it -> humans).
TARGET, CONF_THRESHOLD = 0.95, None
print("(1) CALIBRATION on a labeled validation set:")
print(f"    threshold | auto-accepted | MEASURED accuracy   (target {TARGET:.0%})")
for thr in (0.5, 0.7, 0.8, 0.9, 0.95):
    kept = [ok for c, ok in samples if c >= thr]
    acc = sum(kept) / len(kept) if kept else 0.0
    meets = bool(kept) and acc >= TARGET
    if meets and CONF_THRESHOLD is None: CONF_THRESHOLD = thr
    print(f"       {thr:.2f}   |   {len(kept):>2}/{len(samples)}      |   {acc:6.1%}" + ("   <- meets target" if meets else ""))
CONF_THRESHOLD = CONF_THRESHOLD or 0.95     # if nothing met the target, fall back to the strictest cut
print(f"    -> CALIBRATED CONF_THRESHOLD = {CONF_THRESHOLD}  (DERIVED from data, not picked 'by vibes').\n")

# ---- (2) ROUTING with the CALIBRATED threshold ---------------------------------------------
# Per-field confidence routes attention PRECISELY; ONE aggregate number averages a weak field in
# with strong ones and can wave a bad record through.
def route(extracted):
    conf = extracted["confidence"]
    mean = sum(conf.values()) / len(conf)
    flagged = [f for f, c in conf.items() if c < CONF_THRESHOLD]
    print(f"      per-field conf = {conf}")
    print(f"      aggregate(mean) = {mean:.2f}   AUTO-ACCEPT = {[f for f in conf if f not in flagged]}"
          f"   HUMAN REVIEW = {flagged}")

print(f"(2) ROUTING live extractions at the calibrated threshold ({CONF_THRESHOLD}):")
print("    CLEAN source:"); route(extract("Refund $337.50 on order A-1001, dated 2024-03-03."))
print("    MIXED source:"); route(extract("Refund $337.50 on order A-1001 -- I'm not sure of the exact date, sometime last month?"))
print("    -> The mixed source is confident on amount/order_id but unsure on the date; per-field "
      "confidence flags the\n       weak field for review, while one aggregate mean would average it "
      "away and pass the record.\n")

# ---- (3) AGGREGATE accuracy hides a weak SEGMENT -> stratify by type/field ------------------
segments = {"typed_invoices": (0.99, 400), "handwritten_notes": (0.70, 40), "faxed_scans": (0.98, 160)}
overall = sum(acc * n for acc, n in segments.values()) / sum(n for _, n in segments.values())
print(f"(3) SEGMENT check -- aggregate accuracy = {overall:.1%}  <- looks great in one number...")
for seg, (acc, n) in segments.items():
    print(f"      {seg:18} {acc:.0%} (n={n})" + ("   <-- WEAK SEGMENT: keep human review" if acc < 0.90 else ""))
print("    -> Calibrate the threshold on labeled data, route per FIELD, and stratify by document "
      "TYPE/FIELD before\n       automating -- a single aggregate can hide both a weak field and a weak "
      "segment. (D5.5)")


**How it actually works — two things worth pinning down.**

- **No function computes `confidence` — the model self-reports it.** `extract()` forces a
  `tool_use` with `tool_choice` and just reads `b.input`; we never execute a function or return a
  `tool_result`. The "tool" is **structured output** (D4.3/D4.4): a strict JSON shape Claude must
  fill, `confidence` included. So each score is the model's own **introspection** on *that* text
  (nudged by the tool description) — not a measured quantity. That's exactly why it can't be trusted
  raw, and why we calibrate.
- **Calibration tunes the *threshold*, not the model.** The model never sees the validation results —
  each `extract()` call is **stateless** and assigns confidence the same way every time. The *only*
  thing the routing step consumes from the whole calibration stage is the single number
  `CONF_THRESHOLD`. We **measured** where the model's raw scores actually track accuracy ("≥0.8 →
  ~100% correct *here*") and moved our decision line there. Like calibrating a slightly-off
  thermometer: you don't fix the thermometer, you learn where its readings line up with truth.
  Re-run as data drifts — the threshold only holds for the kind of inputs you measured.

**The anti-patterns (exam distractors).** From the mapping: *trust 97% aggregate.* As
code:

In [ ]:
# ANTI-PATTERN 1: trust the AGGREGATE accuracy and automate everything.
#   if overall_accuracy >= 0.95: drop_human_review()   # the 70% handwritten segment slips through
#   -> Validate accuracy by document TYPE and FIELD before reducing review.

# ANTI-PATTERN 2: one confidence number for the WHOLE extraction.
#   if extraction.confidence >= 0.9: auto_accept()     # a weak 'date' hides behind a strong 'name'
#   -> Emit FIELD-LEVEL confidence and route per field.

# ANTI-PATTERN 3: UNIFORM random sampling for QA.
#   sample = random.sample(all_extractions, 100)       # misses novel errors in small segments
#   -> Use STRATIFIED sampling within each type/field.

# ANTI-PATTERN 4: trust raw model confidence without CALIBRATION.
#   threshold = 0.9   # picked by vibes, not validated against labeled data
#   -> Calibrate thresholds using a labeled validation set.
print("Correct: field-level confidence + stratified sampling + per-segment accuracy, calibrated on "
      "labeled data; route low-confidence/contradictory items to human review.")

**In your own code — you already built this.** Anchor:

- **Exercise 3:** `ccaf-prep/exercises/03-extraction-pipeline/extract.py` — the `record_refund_request`
  tool schema requires a **per-field `confidence` object** (**lines 66–81**);
  `route_by_confidence` (**lines 218–230**) routes each field by its own score against
  `CONF_THRESHOLD` (**line 143**) and prints `aggregate(mean) <- can HIDE a weak field`
  (**line 228**) — exactly this task statement. The few-shot (**lines 101–112**) teaches a
  **high** confidence on a clean message and a **low** one when the source is vague.

Open `extract.py:218` and map `flagged` to "route low-confidence to human review".

**Self-check** (cover the answers)

1. Your extractor reports 97% aggregate accuracy. Why is that not enough to drop human review?
2. One confidence score per extraction, or per field? Why?
3. How should you sample high-confidence extractions for ongoing QA — uniformly or how?
4. Two things route an extraction to human review.

<details><summary>answers</summary>

1. The aggregate can **mask a weak segment** (a document type or field at 70%). Validate **by type and field** first.
2. **Per field** — a weak `date` hides behind a strong `name` in a single combined score.
3. **Stratified** random sampling (within each type/field) — to catch novel errors in small segments.
4. **Low model confidence** and **ambiguous/contradictory source documents**.
</details>

---
## 5.6 · Preserve information provenance and handle uncertainty in multi-source synthesis

> **Task Statement 5.6: Preserve information provenance and handle uncertainty in multi-source synthesis**
>
> **Knowledge of:**
> - How source attribution is lost during summarization steps when findings are compressed without preserving claim-source mappings
> - The importance of structured claim-source mappings that the synthesis agent must preserve and merge when combining findings
> - How to handle conflicting statistics from credible sources: annotating conflicts with source attribution rather than arbitrarily selecting one value
> - Temporal data: requiring publication/collection dates in structured outputs to prevent temporal differences from being misinterpreted as contradictions
>
> **Skills in:**
> - Requiring subagents to output structured claim-source mappings (source URLs, document names, relevant excerpts) that downstream agents preserve through synthesis
> - Structuring reports with explicit sections distinguishing well-established findings from contested ones, preserving original source characterizations and methodological context
> - Completing document analysis with conflicting values included and explicitly annotated, letting the coordinator decide how to reconcile before passing to synthesis
> - Requiring subagents to include publication or data collection dates in structured outputs to enable correct temporal interpretation
> - Rendering different content types appropriately in synthesis outputs—financial data as tables, news as prose, technical findings as structured lists—rather than converting everything to a uniform format

**Plain-English unpacking**

| Guide phrase | What it means concretely |
|---|---|
| Attribution lost in summarization | Compress findings without keeping **claim → source** links and you can no longer say *who* said *what*. |
| **Claim-source mappings** | Each claim carries its source (URL/doc + excerpt); synthesis must **preserve and merge** these, not drop them. |
| Conflicts → **annotate**, don't pick | Two credible sources disagree → keep **both** with attribution; don't silently choose one value. |
| **Dates** prevent false contradictions | 37.0M (2023) vs 41.0M (2018) may differ by **date/methodology**, not error — require publication/collection dates. |
| Well-established vs contested | Structure the report to separate solid findings from disputed ones, preserving each source's characterization. |
| Render by content type | Financial data as **tables**, news as **prose**, technical findings as **lists** — don't flatten everything. |

**Run it — synthesize a conflict, provenance preserved (REAL calls).** Two findings give
**conflicting** Tokyo population figures with different **sources and dates**. With the
provenance instruction, Claude keeps **both** with attribution and **annotates** the
conflict (noting the dates differ); the naive prompt often silently **picks one**, losing
the attribution. The findings are fixtures; the synthesis is a real model decision. (D5.6.)

In [ ]:
# 5.6 — REAL calls: synthesize a CONFLICT. Preserve sources+dates and ANNOTATE, don't pick one.
FINDINGS = [
    {"claim": "Tokyo metro population is about 37.0 million", "value": "37.0M",
     "source": "web:worldstats.example", "date": "2023-01"},
    {"claim": "Tokyo metro population is about 41.0 million", "value": "41.0M",
     "source": "doc:UN-WUP-2018.pdf p.12", "date": "2018-05"}]

PROVENANCE = ("Synthesize a 1-2 sentence briefing from the findings JSON. PRESERVE each claim's "
              "source and date. If two sources give CONFLICTING values, ANNOTATE the conflict "
              "citing BOTH sources and dates — do NOT silently choose one value.")

def synth(instruction):
    r = client.messages.create(model=MODEL, max_tokens=240, system=instruction,
        messages=[{"role": "user", "content": json.dumps(FINDINGS, indent=2)}])
    return "".join(b.text for b in r.content if b.type == "text").strip()

print("WITH provenance instruction:\n ", synth(PROVENANCE))
print("\nANTI-PATTERN (asks for ONE figure):\n ",
      synth("What is Tokyo's metro population? Answer in one short sentence with a single figure."))
print("\n-> The provenance-aware synthesis keeps BOTH 37.0M [2023] and 41.0M [2018] with sources and "
      "flags the conflict (different dates -> maybe not a true contradiction). The naive prompt asks "
      "for ONE\n   figure, so it picks a single value (37.0M) and drops the other source. (D5.6)")

**The anti-patterns (exam distractors).** From the mapping: *arbitrarily pick one value.*
As code:

In [ ]:
# ANTI-PATTERN 1: arbitrarily PICK one of two conflicting values.
#   population = max(values)   # or values[0] — silently discards a credible source
#   -> Annotate the conflict with BOTH sources + dates; let the coordinator reconcile.

# ANTI-PATTERN 2: summarize findings WITHOUT claim-source mappings.
#   summary = summarize([f["claim"] for f in findings])   # sources dropped -> attribution lost
#   -> Carry source (URL/doc + excerpt) through synthesis with each claim.

# ANTI-PATTERN 3: omit DATES, then treat a temporal gap as a contradiction.
#   "Sources disagree: 37M vs 41M (error?)"   # actually 2023 vs 2018 — different snapshots
#   -> Require publication/collection dates so time differences aren't misread as errors.

# ANTI-PATTERN 4: flatten every content type into uniform prose.
#   report = to_prose(financial_table)   # numbers belong in a table, not a paragraph
#   -> Render financial data as tables, news as prose, technical findings as lists.
print("Correct: preserve claim-source mappings through synthesis; annotate conflicts with sources "
      "+ dates (never pick arbitrarily); require dates so temporal gaps aren't read as contradictions.")

**In your own code — you already built this.** Anchor:

- **Exercise 4:** `ccaf-prep/exercises/04-multi-agent-research/coordinator.py` — the toy sources
  (**lines 51–61**) each carry their **own `source` + `date`**, and one population figure
  **deliberately conflicts** (37.0M web 2023 vs 41.0M doc 2018). The synthesis system prompt
  (**lines 214–216**) orders the agent to PRESERVE each `[source]` and ANNOTATE conflicts;
  the provenance report (**lines 226–238**) prints the claim→source map and flags the
  population **CONFLICT** "ANNOTATED, not resolved (… dates differ so it may not be a true
  contradiction)".

Run `coordinator.py` and read the `[PROVENANCE]` block + the `! CONFLICT` line.

**Self-check** (cover the answers)

1. Two credible sources report 37M and 41M. What should synthesis do with the disagreement?
2. Why require publication/collection dates in every structured finding?
3. What is a "claim-source mapping" and what step most often destroys it?
4. How should financial data vs news be rendered in a synthesis report?

<details><summary>answers</summary>

1. **Annotate** the conflict with **both** sources and dates — do not arbitrarily pick one value.
2. So a **temporal** difference (2023 vs 2018) isn't misread as a **contradiction** — different snapshots, not an error.
3. A claim paired with its **source** (URL/doc + excerpt); **summarization** destroys it when findings are compressed without preserving the link.
4. Financial data as a **table**, news as **prose** (technical findings as lists) — don't flatten everything to a uniform format.
</details>

---
## Coming next / related domains

You've made Domain 5 observable: a **case-facts block** beats progressive summarization
(5.1), explicit **escalation criteria** beat sentiment (5.2), **structured error context**
enables coordinator recovery (5.3), **subagent delegation + manifests** survive long
explorations and crashes (5.4), **field-level confidence** beats a deceptive aggregate
(5.5), and **claim-source mappings + dates** keep provenance through synthesis (5.6).

**Threads that continue elsewhere:**
- **D1.2 / D1.3** — the hub-and-spoke + `Task` delegation behind 5.3/5.4 live in
  `notebooks/D1_agentic_loops.ipynb` §1.2–1.3 and `exercises/04-multi-agent-research/coordinator_sdk.py`.
- **D2.2** — the `isError` / structured-error *shape* (transient/validation/business) that
  5.3 propagates across agents is built in `notebooks/D2_tool_design_mcp.ipynb` §2.2.
- **D4.3 / D4.4** — the forced-tool extraction and validation-retry loop that 5.5's
  confidence routing sits on top of → Domain 4 (`exercises/03-extraction-pipeline/extract.py`).

Per the study order (`README.md`): D5 unlocks the first exercise wave — **Ex1** then
**Ex4** (both D1+D2+D5).

## Sample exam questions — Domain 5
<!-- ccaf:closing -->

Official sample questions whose tested skill lives in this domain. Cover the answer, say why each of the other three is a distractor, then reveal. (You'll meet them again, grouped by scenario, in `mock_exam_and_review.ipynb`.)

**Question 8.** The web search subagent times out while researching a complex topic. You need to design how this failure information flows back to the coordinator agent. Which error propagation approach best enables intelligent recovery?

- **A)** Return structured error context to the coordinator including the failure type, the attempted query, any partial results, and potential alternative approaches.
- **B)** Implement automatic retry logic with exponential backoff within the subagent, returning a generic "search unavailable" status only after all retries are exhausted.
- **C)** Catch the timeout within the subagent and return an empty result set marked as successful.
- **D)** Propagate the timeout exception directly to a top-level handler that terminates the entire research workflow.

<sub>Maps to: D5 §5.3 (error propagation across multi-agent systems)</sub>

<details><summary>Answer + explanation</summary>

**Correct answer: A**

Structured error context gives the coordinator the information it needs to make intelligent recovery decisions—whether to retry with a modified query, try an alternative approach, or proceed with partial results. Option B's generic status hides valuable context from the coordinator, preventing informed decisions. Option C suppresses the error by marking failure as success, which prevents any recovery and risks incomplete research outputs. Option D terminates the entire workflow unnecessarily when recovery strategies could succeed.

</details>

---

**Question 3.** Your agent achieves 55% first-contact resolution, well below the 80% target. Logs show it escalates straightforward cases (standard damage replacements with photo evidence) while attempting to autonomously handle complex situations requiring policy exceptions. What's the most effective way to improve escalation calibration?

- **A)** Add explicit escalation criteria to your system prompt with few-shot examples demonstrating when to escalate versus resolve autonomously.
- **B)** Have the agent self-report a confidence score (1-10) before each response and automatically route requests to humans when confidence falls below a threshold.
- **C)** Deploy a separate classifier model trained on historical tickets to predict which requests need escalation before the main agent begins processing.
- **D)** Implement sentiment analysis to detect customer frustration levels and automatically escalate when negative sentiment exceeds a threshold.

<sub>Maps to: D5 §5.2 (escalation calibration) — spans two domains (also D4 §4.1)</sub>

<details><summary>Answer + explanation</summary>

**Correct answer: A**

Adding explicit escalation criteria with few-shot examples directly addresses the root cause: unclear decision boundaries. This is the proportionate first response before adding infrastructure. Option B fails because LLM self-reported confidence is poorly calibrated—the agent is already incorrectly confident on hard cases. Option C is over-engineered, requiring labeled data and ML infrastructure when prompt optimization hasn't been tried. Option D solves a different problem entirely; sentiment doesn't correlate with case complexity, which is the actual issue.

</details>

---

## Exercises that use this domain

Exercises are **cross-domain** — none is D5-only — so do them once all their domains are studied
(see [`README.md`](./README.md) for the wave order).

| Exercise | Domains it reinforces | Do it after you've studied |
|----------|-----------------------|-----------------------------|
| **Ex1** `../exercises/01-support-agent/`        | D1 + D2 + D5 | D1, D2, **D5** |
| **Ex4** `../exercises/04-multi-agent-research/` | D1 + D2 + D5 | D1, D2, **D5** |
| **Ex3** `../exercises/03-extraction-pipeline/`  | D4 + D5      | D5, **D4**     |

D5 sits at the center of the reliability story: it **unlocks the first exercise wave** — after
D5 you can do **Ex1** then **Ex4** (both D1+D2+D5). **Ex3** (D4+D5) waits on **D4**.